In [21]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import make_scorer, recall_score, precision_score, f1_score


In [22]:
# Load clean dataset 

processed_dir = Path.home() / "Documents" / "diabetes-risk-project" / "data" / "processed"

df_clean = pd.read_csv(processed_dir / "clean_project_dataset.csv")

print(df_clean.shape)
df_clean.head()

(9232, 37)


,SEQN,RIDAGEYR,RIAGENDR,RIDRETH3,DMDEDUC2,INDFMPIR,BMXBMI,BMXWAIST,BPXOSY1,BPXOSY2,...,ALQ111,ALQ121,ALQ130,DBQ700,DBD895,DBD900,DBD905,DBD910,doctor_diabetes,diabetes
0,109266.0,29.0,2.0,6.0,5.0,5.00,37.8,117.9,99.0,99.0,...,1.0,1.000000e+01,1.0,3.0,7.000000e+00,5.397605e-79,5.397605e-79,5.000000e+00,0.0,0
1,109267.0,21.0,2.0,2.0,4.0,5.00,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1.0,4.000000e+00,5.397605e-79,5.397605e-79,5.397605e-79,0.0,0
2,109271.0,49.0,1.0,3.0,2.0,NaN,29.7,120.4,102.0,108.0,...,1.0,5.397605e-79,NaN,3.0,2.000000e+00,2.000000e+00,5.397605e-79,5.397605e-79,0.0,0
3,109273.0,36.0,1.0,3.0,4.0,0.83,21.9,86.8,116.0,110.0,...,1.0,5.397605e-79,NaN,4.0,2.000000e+00,2.000000e+00,5.397605e-79,7.000000e+00,0.0,0
4,109274.0,68.0,1.0,7.0,4.0,1.20,30.2,109.6,138.0,132.0,...,1.0,4.000000e+00,2.0,2.0,5.397605e-79,NaN,5.397605e-79,5.397605e-79,1.0,1


In [23]:
#Check the target distribution 
print("Target Distribution")
print(df_clean["diabetes"].value_counts())
print("Target Distribution Percentage")
print(df_clean["diabetes"].value_counts(normalize=True)*100)

Target Distribution
diabetes
0    7412
1    1820
Name: count, dtype: int64
Target Distribution Percentage
diabetes
0    80.285962
1    19.714038
Name: proportion, dtype: float64


In [24]:
# Target leakage variables 
leakage_cols = ["SEQN", "diabetes", "doctor_diabetes", "DIQ010", "LBXGH", "LBXGLU"]
leakage_cols = [col for col in leakage_cols if col in df_clean.columns]
X=df_clean.drop(columns=leakage_cols)
y=df_clean["diabetes"]
print("predictor dataset shape:", X.shape)
print("Target shape:", y.shape)

print("\n Predictor columns:")
print(X.columns.tolist())

predictor dataset shape: (9232, 31)
Target shape: (9232,)

 Predictor columns:
['RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'DMDEDUC2', 'INDFMPIR', 'BMXBMI', 'BMXWAIST', 'BPXOSY1', 'BPXOSY2', 'BPXOSY3', 'BPXODI1', 'BPXODI2', 'BPXODI3', 'BPQ020', 'BPQ080', 'SMQ020', 'PAQ650', 'PAQ655', 'PAD660', 'PAQ665', 'PAQ670', 'PAD675', 'PAD680', 'ALQ111', 'ALQ121', 'ALQ130', 'DBQ700', 'DBD895', 'DBD900', 'DBD905', 'DBD910']


In [25]:
#define the categorical and Numerical variables 
categorical_features = [
    "RIAGENDR",
    "RIDRETH3",
    "DMDEDUC2",
    "BPQ020",
    "BPQ080",
    "SMQ020",
    "PAQ650",
    "PAQ665",
    "ALQ111",
    "ALQ121",
    "DBQ700"
]
categorical_features = [col for col in categorical_features if col in X.columns]
numeric_features = [col for col in X.columns if col not in categorical_features]
print("Categorical features:")
print(categorical_features)
print("Numeric features:")
print(numeric_features)

Categorical features:
['RIAGENDR', 'RIDRETH3', 'DMDEDUC2', 'BPQ020', 'BPQ080', 'SMQ020', 'PAQ650', 'PAQ665', 'ALQ111', 'ALQ121', 'DBQ700']
Numeric features:
['RIDAGEYR', 'INDFMPIR', 'BMXBMI', 'BMXWAIST', 'BPXOSY1', 'BPXOSY2', 'BPXOSY3', 'BPXODI1', 'BPXODI2', 'BPXODI3', 'PAQ655', 'PAD660', 'PAQ670', 'PAD675', 'PAD680', 'ALQ130', 'DBD895', 'DBD900', 'DBD905', 'DBD910']


In [26]:
#preprocessing pipelines 
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [27]:
#Define quick learnability models
dummy_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent"))
])

logistic_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

random_forest_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

models = {
    "Dummy Classifier": dummy_model,
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model
}

In [28]:
#Cross-validation setup and scoring metrics
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "recall": make_scorer(recall_score),
    "precision": make_scorer(precision_score, zero_division=0),
    "f1": make_scorer(f1_score)
}

In [29]:
#Run cross-validation
results = []

for model_name, model in models.items():
    print(f"Running cross-validation for: {model_name}")
    
    cv_results = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )
    
    row = {"model": model_name}
    
    for metric in scoring.keys():
        scores = cv_results[f"test_{metric}"]
        row[f"{metric}_mean"] = scores.mean()
        row[f"{metric}_std"] = scores.std()
    
    results.append(row)

learnability_results = pd.DataFrame(results)

learnability_results

Running cross-validation for: Dummy Classifier
Running cross-validation for: Logistic Regression
Running cross-validation for: Random Forest


,model,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,recall_mean,recall_std,precision_mean,precision_std,f1_mean,f1_std
0,Dummy Classifier,0.802860,0.000052,0.500000,0.000000,0.500000,0.000000,0.197140,0.000052,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,Logistic Regression,0.716532,0.008266,0.730815,0.009822,0.803484,0.007206,0.471740,0.018452,0.754396,0.019704,0.387636,0.008991,0.512043,0.010644
2,Random Forest,0.790729,0.006233,0.685619,0.005349,0.802454,0.004667,0.483901,0.017723,0.512088,0.009611,0.472051,0.013709,0.491114,0.008828
